# Application du solver au dataset

## Breast Cancer

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

df_BC=pd.read_csv("breastcancer_processed.csv")
df_BC.head()
TARGET_COL = "Benign"
X = df_BC.drop(columns=[TARGET_COL])
y = df_BC[TARGET_COL]

print("Shape X:", X.shape, "| Shape y:", y.shape)
print("Répartition de la cible:\n", y.value_counts())

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

logreg = LogisticRegression(
    solver="liblinear",
    max_iter=500,
    class_weight="balanced",
    random_state=42
)
logreg.fit(X_train, y_train)

y_pred = logreg.predict(X_test)

print("\nAccuracy (test):", round(accuracy_score(y_test, y_pred), 4))
print("\nRapport de classification (test):")
print(classification_report(y_test, y_pred, digits=3))

coef = pd.Series(logreg.coef_[0], index=X.columns).sort_values(ascending=False)
print("\nIntercept:", float(logreg.intercept_[0]))
print("\nCoefficients (beta) :")
display(coef.to_frame(name="beta"))


Shape X: (683, 9) | Shape y: (683,)
Répartition de la cible:
 Benign
0    444
1    239
Name: count, dtype: int64

Accuracy (test): 0.9649

Rapport de classification (test):
              precision    recall  f1-score   support

           0      0.991     0.955     0.972       111
           1      0.922     0.983     0.952        60

    accuracy                          0.965       171
   macro avg      0.956     0.969     0.962       171
weighted avg      0.967     0.965     0.965       171


Intercept: -5.769047483353211

Coefficients (beta) :


,beta
BareNuclei,0.492783
UniformityOfCellShape,0.312884
UniformityOfCellSize,0.290715
ClumpThickness,0.231480
Mitoses,0.194093
NormalNucleoli,0.150516
MarginalAdhesion,0.140185
BlandChromatin,0.004105
SingleEpithelialCellSize,-0.009519


In [6]:
import numpy as np
from gurobipy import Model, GRB, quicksum

TARGET = "Benign"
X = df_BC.drop(columns=[TARGET])

beta = logreg.coef_[0]
features = list(X.columns)

# Choisir x (préduit 1) et y (préduit 0)
pred = logreg.predict(X)
ix = X.index[pred == 1][0]
iy = X.index[pred == 0][0]
x = X.loc[ix]
y_ = X.loc[iy]

# Vérifier p(x) > p(y) et s(x) > s(y)
px = float(logreg.predict_proba([x])[0, 1])
py = float(logreg.predict_proba([y_])[0, 1])
sx = float(logreg.decision_function([x])[0])
sy = float(logreg.decision_function([y_])[0])

print(f"x index={ix} | p(x)={px:.4f} | s(x)={sx:.4f}")
print(f"y index={iy} | p(y)={py:.4f} | s(y)={sy:.4f}")
assert sx > sy, "On veut comparer un x strictement préféré à y (s(x) > s(y))."

# Contributions Δ_j = β_j (x_j - y_j)
delta = {f: float(beta[i] * (x[f] - y_[f])) for i, f in enumerate(features)}
pros = [f for f in features if delta[f] > 0]
cons = [f for f in features if delta[f] < 0]
print("s(x)-s(y) =", sum(delta.values()))
print("pros:", pros)
print("cons:", cons)

# Exemple : explication (1-1) par MILP
T = [(p, c) for p in pros for c in cons if delta[p] + delta[c] > 0]

m = Model("explain_1-1_realdata"); m.params.OutputFlag = 0
z = m.addVars(T, vtype=GRB.BINARY)

for c in cons:
    m.addConstr(quicksum(z[p, c] for p in pros if (p, c) in z) == 1)

m.setObjective(quicksum(z[p, c] for (p, c) in T), GRB.MINIMIZE)
m.optimize()

if m.Status == GRB.OPTIMAL:
    E = [(p, c) for (p, c) in T if z[p, c].X > 0.5]
    print("Explication (1-1):", E)
else:
    print("Pas d'explication (1-1) (infeasible).")


x index=1 | p(x)=0.9796 | s(x)=3.8723
y index=0 | p(y)=0.0458 | s(y)=-3.0372
s(x)-s(y) = 6.909508214255732
pros: ['UniformityOfCellSize', 'UniformityOfCellShape', 'MarginalAdhesion', 'BareNuclei', 'NormalNucleoli']
cons: ['SingleEpithelialCellSize']
Set parameter Username


Set parameter LicenseID to value 2755089
Academic license - for non-commercial use only - expires 2026-12-15
Explication (1-1): [('NormalNucleoli', 'SingleEpithelialCellSize')]


c:\Users\fanny\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\fanny\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\fanny\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
c:\Users\fanny\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LogisticRegression was fitted with feature names
  warnings.warn(
